# Arsenal YouTube + Naver Cafe Sentiment Pipeline
#
Local Jupyter용입니다.
- 통합코드 0~4단계 구조 유지
- YouTube는 이미 저장된 감정분석 엑셀 파일 사용
- video_title / video_published_at / channel_title 없으면 YouTube API로 추가
- 네이버 카페 크롤링은 "본문+댓글" 방식 사용
- 중간 저장/재시작 파일을 남겨서 안전하게 구성



In [ ]:
# =====================================================
# 0. 패키지 설치/확인
# =====================================================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("openpyxl", "openpyxl"),
    ("tqdm", "tqdm"),
    ("google-api-python-client", "googleapiclient"),
    ("openai", "openai"),
    ("selenium", "selenium"),
    ("webdriver-manager", "webdriver_manager"),
    ("beautifulsoup4", "bs4"),
]

for pip_name, import_name in REQUIRED_PACKAGES:
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

print("패키지 확인 완료")



In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

import os
import re
import json
import time
import shutil
import traceback
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

from googleapiclient.discovery import build
from openai import OpenAI

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)



In [ ]:
import os

# =====================================================
# 2. 기본 설정
# =====================================================

# API KEY를 입력하세요.
# YouTube 영상 제목/게시일을 붙일 때 YOUTUBE_API_KEY 사용
# GPT 2차 감정분석에서 OPENAI_API_KEY 사용
YOUTUBE_API_KEY = os.environ["YOUTUBE_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

OPENAI_MODEL = "gpt-4.1-mini"

# 사용자 환경 경로
BASE_DIR = Path(r"C:\Users\soyou\OneDrive\바탕 화면\textdata")
OUTPUT_DIR = BASE_DIR / "arsenal_pipeline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 이미 저장된 YouTube 감정분석 엑셀 파일 사용
YOUTUBE_LABELED_XLSX = BASE_DIR / "arsenal_gpt_all_labeled_batch_neutral_refined.xlsx"

# 카페 크롤링 시작 글 URL
START_ARTICLE_URL = (
    "https://cafe.naver.com/f-e/cafes/11643699/articles/526375"
)


CLUB = "Arsenal"
CLUB_KO = "아스날"
ROUND = "38R"

MATCH_38R = datetime(2026, 5, 25, 0, 0, 0)
START_DATE = datetime(2026, 5, 22, 0, 0, 0)
END_DATE = datetime(2026, 5, 27, 23, 59, 59)
BASE_YEAR = 2026

MAX_CHECK_POSTS = 2000
RESTART_EVERY = 70
BATCH_SIZE = 20

# 크롤링 중간 저장/재시작 파일
CAFE_CHECKPOINT_CSV = OUTPUT_DIR / "arsenal_cafe_checkpoint.csv"
CAFE_PROGRESS_JSON = OUTPUT_DIR / "arsenal_cafe_progress.json"
CAFE_FINAL_CSV = OUTPUT_DIR / "arsenal_cafe_posts_comments.csv"
CAFE_FINAL_XLSX = OUTPUT_DIR / "arsenal_cafe_posts_comments.xlsx"

# YouTube 변환/통합 저장 파일
YOUTUBE_WITH_META_CSV = OUTPUT_DIR / "arsenal_youtube_labeled_with_video_meta.csv"
YOUTUBE_WITH_META_XLSX = OUTPUT_DIR / "arsenal_youtube_labeled_with_video_meta.xlsx"
YOUTUBE_FINAL_CSV = OUTPUT_DIR / "arsenal_youtube_final_for_merge.csv"
MERGED_BEFORE_GPT_CSV = OUTPUT_DIR / "arsenal_youtube_cafe_merged_before_gpt.csv"
REANALYSIS_CHECKPOINT_XLSX = OUTPUT_DIR / "arsenal_reanalysis_checkpoint.xlsx"

OUTPUT_FINAL_CSV = OUTPUT_DIR / "arsenal_youtube_cafe_after_labeled.csv"
OUTPUT_FINAL_XLSX = OUTPUT_DIR / "arsenal_youtube_cafe_after_labeled.xlsx"

# 통합코드 FINAL_COLUMNS + YouTube 원본 메타 컬럼 추가
# video_title -> title에도 복사됩니다.
# video_published_at -> content_published_at에도 복사됩니다.
# channel_title은 참고용으로 보존합니다.
FINAL_COLUMNS = [
    "source",
    "round",
    "club",
    "title",
    "video_id",
    "video_title",
    "video_published_at",
    "channel_title",
    "content_type",
    "text_original",
    "text_clean",
    "content_published_at",
    "comment_published_at",
    "like_count",
    "url",
    "period",
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason",
    "topic",
]

CAFE_COLUMNS = [
    "source", "round", "title", "post_date", "comment_date",
    "text_original", "text_clean", "content_type", "period", "url"
]

print("OUTPUT_DIR:", OUTPUT_DIR)
print("수집 기간:", START_DATE, "~", END_DATE)



In [ ]:
# =====================================================
# 3. 공통 유틸 함수
# =====================================================

def api_key_is_set(value):
    value = str(value or "").strip()
    return bool(value) and "여기에" not in value


def get_video_id(x):
    x = str(x).strip()
    patterns = [
        r"v=([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)",
        r"embed/([^?&]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, x)
        if m:
            return m.group(1)
    return x


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"&[a-zA-Z]+;", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_sentiment(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    replace_map = {
        "긍정": "positive",
        "중립": "neutral",
        "부정": "negative",
    }
    return replace_map.get(x, x)


def to_naive_datetime_series(s, utc=False):
    if utc:
        return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)
    return pd.to_datetime(s, errors="coerce")


def to_naive_timestamp(x):
    ts = pd.to_datetime(x, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    if isinstance(ts, pd.Timestamp) and ts.tzinfo is not None:
        return ts.tz_convert(None)
    return ts


def get_period(dt):
    ts = to_naive_timestamp(dt)
    if pd.isna(ts):
        return "unknown"
    if ts.to_pydatetime() < MATCH_38R:
        return "before_38R"
    return "after_38R"


def parse_json_array(text):
    text = str(text).strip()
    text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()
    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


def first_existing_col(df, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"필수 컬럼이 없습니다. 후보: {candidates}, 현재 컬럼: {df.columns.tolist()}")
    return None


def safe_read_excel(path):
    path = Path(path)
    try:
        return pd.read_excel(path)
    except PermissionError:
        copied = OUTPUT_DIR / f"_copy_{path.name}"
        shutil.copy2(path, copied)
        print("원본 엑셀이 열려 있어 복사본으로 읽었습니다:", copied)
        return pd.read_excel(copied)


def save_dataframe_both(df, csv_path, xlsx_path=None):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if xlsx_path is not None:
        xlsx_path = Path(xlsx_path)
        df.to_excel(xlsx_path, index=False)



In [ ]:
# =====================================================
# 4. match_context 설정(match context 1차에 넣고 돌려서 틀만 유지-2차분석 위해)
# =====================================================

DEFAULT_MATCH_CONTEXT = """

[아스날 선수단 정보]
감독: 미켈 아르테타

선수:
가브리엘 마갈량이스, 가브리엘 마르티넬리, 가브리엘 제주스,
노니 마두에케, 다비드 라야, 데클란 라이스, 레안드로 트로사르,
리카르도 칼라피오리, 마르틴 수비멘디, 마르틴 외데고르,
마일스 루이스-스켈리, 막스 다우만, 말리 샐먼, 미켈 메리노,
벤 화이트, 부카요 사카, 빅토르 요케레스, 안드레 해리먼-아누스,
에베레치 에제, 위리엔 팀버르, 윌리엄 살리바, 이선 은와네리,
카이 하베르츠, 케파 아리사발라가, 크리스티안 뇌가르,
크리스티안 모스케라, 토미 셋포드, 피에로 인카피에, 야쿠브 키비오르

[경기 맥락]
프리미어리그 38라운드: 아스날 2:1 승리. 아스날과 상대팀 모두 득점한 경기.
프리미어리그 37라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 36라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 35라운드: 아스날 3:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 34라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 33라운드: 아스날 1:2 패배. 아스날이 패배한 경기.
프리미어리그 32라운드: 아스날 1:2 패배. 아스날이 패배한 경기.
프리미어리그 30라운드: 아스날 2:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 29라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.
프리미어리그 28라운드: 아스날 2:1 승리. 아스날과 상대팀 모두 득점한 경기.
UEFA 챔피언스리그 결승: 아스날 1:1 무승부 후 승부차기 패배. 아스날이 준우승한 경기이며, 조롱·비꼼·아쉬움이 포함될 수 있는 영상.

[판단 기준]
아스날 선수, 감독, 전술, 경기 결과, 구단 운영에 대한 칭찬은 positive.
아스날 선수, 감독, 전술, 경기 결과, 구단 운영에 대한 비판/조롱/불만은 negative.
상대팀, 방송진, 심판, 리그만 언급하면 아스날 감정이 아니므로 neutral 또는 해당 target으로 둔다.
""".strip()


def make_match_context(row):
    source = row.get("source", "")
    title = row.get("title", "")
    period = row.get("period", "")
    content_type = row.get("content_type", "")
    content_date = row.get("content_published_at", "")
    comment_date = row.get("comment_published_at", "")

    return f"""
분석 대상 클럽: {CLUB_KO}({CLUB})
라운드: {row.get("round", ROUND)}
기간 구분: {period}
데이터 출처: {source}
콘텐츠 유형: {content_type}
제목/맥락: {title}
콘텐츠 게시일: {content_date}
댓글 게시일: {comment_date}
기본 경기 맥락: {DEFAULT_MATCH_CONTEXT}
""".strip()



In [ ]:
# =====================================================
# 5. YouTube 데이터: 이미 저장된 엑셀 파일 사용
#    - 댓글 API 수집/최초 감정분석은 스킵
#    - video_title / video_published_at / channel_title 없으면 추가
# =====================================================

yt = safe_read_excel(YOUTUBE_LABELED_XLSX)

print("YouTube 엑셀 shape:", yt.shape)
print("YouTube 엑셀 컬럼:")
print(yt.columns.tolist())

if "video_id" not in yt.columns and "video_url" in yt.columns:
    yt["video_id"] = yt["video_url"].apply(get_video_id)

if "gpt_sentiment" in yt.columns:
    print("\n현재 gpt_sentiment 분포:")
    print(yt["gpt_sentiment"].value_counts(dropna=False))

if "gpt_sentiment_1st" in yt.columns:
    print("\n1차 gpt_sentiment_1st 분포:")
    print(yt["gpt_sentiment_1st"].value_counts(dropna=False))

display(yt.head())



끊긴거 복구

In [ ]:
# 5번: YouTube 저장 엑셀 다시 로드
yt = safe_read_excel(YOUTUBE_LABELED_XLSX)

print("YouTube shape:", yt.shape)
print(yt.columns.tolist())
display(yt.head())

In [ ]:
# =====================================================
# 6. YouTube 영상 메타데이터 추가(컬럼 통합 위해)
# =====================================================

def fill_youtube_video_metadata(df):
    df = df.copy()

    if "video_id" not in df.columns:
        raise KeyError("video_id 컬럼이 필요합니다.")

    need_meta = (
        "video_title" not in df.columns or
        "video_published_at" not in df.columns or
        "channel_title" not in df.columns or
        df.get("video_title", pd.Series(dtype=object)).isna().all()
    )

    if not need_meta:
        print("이미 video_title / video_published_at / channel_title 컬럼이 있습니다.")
        return df

    for col in ["video_title", "video_published_at", "channel_title"]:
        if col not in df.columns:
            df[col] = None

    if not api_key_is_set(YOUTUBE_API_KEY):
        print("YOUTUBE_API_KEY가 없어 API 조회는 스킵합니다.")
        print("video_title은 match_context 또는 video_id로 대체합니다.")
        fallback_title_col = first_existing_col(df, ["match_context", "video_id"], required=False)
        if fallback_title_col:
            df["video_title"] = df["video_title"].fillna(df[fallback_title_col])
        df["video_published_at"] = df["video_published_at"].fillna(pd.NaT)
        df["channel_title"] = df["channel_title"].fillna("unknown")
        return df

    youtube_client = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    video_ids = df["video_id"].dropna().astype(str).unique().tolist()
    video_info = {}

    for start in tqdm(range(0, len(video_ids), 50), desc="YouTube 영상 정보 조회"):
        batch_ids = video_ids[start:start + 50]
        try:
            request = youtube_client.videos().list(
                part="snippet",
                id=",".join(batch_ids),
            )
            response = request.execute()
            items = response.get("items", [])

            for item in items:
                v_id = item.get("id")
                snippet = item.get("snippet", {})
                video_info[v_id] = {
                    "video_title": snippet.get("title"),
                    "video_published_at": snippet.get("publishedAt"),
                    "channel_title": snippet.get("channelTitle"),
                }

            for v_id in batch_ids:
                if v_id not in video_info:
                    video_info[v_id] = {
                        "video_title": "영상 정보 없음",
                        "video_published_at": None,
                        "channel_title": None,
                    }

        except Exception as e:
            print("영상 정보 조회 실패:", batch_ids, e)
            for v_id in batch_ids:
                video_info[v_id] = {
                    "video_title": "영상 정보 조회 실패",
                    "video_published_at": None,
                    "channel_title": None,
                }

        time.sleep(0.1)

    df["video_title"] = df["video_id"].astype(str).map(
        lambda x: video_info.get(x, {}).get("video_title")
    )
    df["video_published_at"] = df["video_id"].astype(str).map(
        lambda x: video_info.get(x, {}).get("video_published_at")
    )
    df["channel_title"] = df["video_id"].astype(str).map(
        lambda x: video_info.get(x, {}).get("channel_title")
    )

    return df


yt = fill_youtube_video_metadata(yt)
save_dataframe_both(yt, YOUTUBE_WITH_META_CSV, YOUTUBE_WITH_META_XLSX)

print("YouTube 메타데이터 저장 완료:")
print(YOUTUBE_WITH_META_CSV)
print(YOUTUBE_WITH_META_XLSX)

display(yt[["video_id", "video_title", "video_published_at", "channel_title"]].drop_duplicates().head())



In [ ]:
# =====================================================
# 7. YouTube 데이터를 통합 컬럼으로 변환(컬럼통합위해)
# =====================================================

def make_youtube_final_from_saved_excel(yt_df):
    yt_df = yt_df.copy()

    original_col = first_existing_col(yt_df, ["comment_original", "text_original", "comment"], required=True)
    clean_col = first_existing_col(yt_df, ["comment_clean", "text_clean", "comment_original"], required=True)
    published_col = first_existing_col(yt_df, ["published_at", "comment_published_at"], required=True)
    url_col = first_existing_col(yt_df, ["video_url", "url"], required=True)

    youtube_final = pd.DataFrame(index=yt_df.index)
    youtube_final["source"] = "youtube"
    youtube_final["round"] = ROUND
    youtube_final["club"] = CLUB
    youtube_final["title"] = yt_df["video_title"].fillna(yt_df.get("match_context", DEFAULT_MATCH_CONTEXT))
    youtube_final["video_id"] = yt_df.get("video_id")
    youtube_final["video_title"] = yt_df.get("video_title")
    youtube_final["video_published_at"] = yt_df.get("video_published_at")
    youtube_final["channel_title"] = yt_df.get("channel_title")
    youtube_final["content_type"] = "video_comment"
    youtube_final["text_original"] = yt_df[original_col]
    youtube_final["text_clean"] = yt_df[clean_col].fillna(yt_df[original_col].apply(clean_text))

    youtube_final["content_published_at"] = to_naive_datetime_series(
        yt_df["video_published_at"],
        utc=True,
    )
    youtube_final["comment_published_at"] = to_naive_datetime_series(
        yt_df[published_col],
        utc=True,
    )

    youtube_final["like_count"] = yt_df["like_count"] if "like_count" in yt_df.columns else None
    youtube_final["url"] = yt_df[url_col]
    youtube_final["period"] = youtube_final["comment_published_at"].apply(get_period)

    # 핵심:
    # 엑셀 파일이 neutral_refined 결과까지 담고 있으면,
    # 카페와 합친 뒤 "유튜브 1차 neutral + 카페 전체"를 다시 돌리기 위해 gpt_sentiment_1st 사용.
    sentiment_col = "gpt_sentiment_1st" if "gpt_sentiment_1st" in yt_df.columns else "gpt_sentiment"
    reason_col = "gpt_reason_1st" if "gpt_reason_1st" in yt_df.columns else "gpt_reason"

    youtube_final["gpt_sentiment"] = yt_df[sentiment_col].apply(normalize_sentiment)
    youtube_final["gpt_confidence"] = yt_df["gpt_confidence"] if "gpt_confidence" in yt_df.columns else None
    youtube_final["gpt_target"] = yt_df["gpt_target"] if "gpt_target" in yt_df.columns else None
    youtube_final["gpt_reason"] = yt_df[reason_col] if reason_col in yt_df.columns else None
    youtube_final["topic"] = None

    youtube_final = youtube_final.reindex(columns=FINAL_COLUMNS)
    return youtube_final


youtube_final = make_youtube_final_from_saved_excel(yt)
save_dataframe_both(youtube_final, YOUTUBE_FINAL_CSV)

print("youtube_final shape:", youtube_final.shape)
print(youtube_final["gpt_sentiment"].value_counts(dropna=False))
display(youtube_final.head())



In [ ]:
# =====================================================
# 8. 네이버 카페 크롤링: 첨부한 본문+댓글 방식
# =====================================================

def parse_cafe_date(date_text):
    if date_text is None or pd.isna(date_text):
        return None

    text = str(date_text).strip()
    text = re.sub(r"\s+", " ", text)

    is_pm = "오후" in text
    is_am = "오전" in text
    text = text.replace("오전", "").replace("오후", "").strip()

    m = re.search(
        r"(\d{4})\.(\d{1,2})\.(\d{1,2})\.?\s*(?:(\d{1,2}):(\d{2}))?",
        text,
    )
    if m:
        year = int(m.group(1))
        month = int(m.group(2))
        day = int(m.group(3))
        hour = int(m.group(4) or 0)
        minute = int(m.group(5) or 0)
        if is_pm and hour < 12:
            hour += 12
        if is_am and hour == 12:
            hour = 0
        return datetime(year, month, day, hour, minute)

    m = re.search(
        r"(\d{1,2})\.(\d{1,2})\.?\s*(?:(\d{1,2}):(\d{2}))?",
        text,
    )
    if m:
        month = int(m.group(1))
        day = int(m.group(2))
        hour = int(m.group(3) or 0)
        minute = int(m.group(4) or 0)
        if is_pm and hour < 12:
            hour += 12
        if is_am and hour == 12:
            hour = 0
        return datetime(BASE_YEAR, month, day, hour, minute)

    return None


parse_article_date = parse_cafe_date
parse_comment_date = parse_cafe_date


def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-dev-shm-usage")
    options.page_load_strategy = "eager"

    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
    except Exception as e:
        print("webdriver-manager 사용 실패, Selenium 기본 Chrome으로 재시도:", e)
        driver = webdriver.Chrome(options=options)

    driver.implicitly_wait(3)
    return driver


def enter_cafe_main(driver):
    try:
        driver.switch_to.default_content()
        driver.switch_to.frame("cafe_main")
        return True
    except Exception as e:
        print("cafe_main iframe 진입 실패:", e)
        return False


def get_article_title(driver):
    selectors = [
        "h3.title_text",
        ".title_text",
        ".ArticleTitle",
        ".article_header .title",
        "h3",
        "h1",
    ]
    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                title = elem.text.strip()
                if title:
                    return title
        except Exception:
            pass
    return ""


def get_article_date(driver):
    selectors = [
        ".date",
        "span.date",
        ".ArticleHeader .date",
        ".article_info .date",
        ".WriterInfo .date",
    ]
    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                date_text = elem.text.strip()
                article_date = parse_article_date(date_text)
                if article_date is not None:
                    return date_text, article_date
        except Exception:
            pass

    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
        m = re.search(r"\d{4}\.\d{1,2}\.\d{1,2}\.?\s*(?:오전|오후)?\s*\d{0,2}:?\d{0,2}", body_text)
        if m:
            date_text = m.group(0)
            return date_text, parse_article_date(date_text)
    except Exception:
        pass

    return "", None


def get_article_content(driver):
    paragraph_texts = []

    paragraph_selectors = [
        ".se-text-paragraph",
        ".se-main-container .se-component-content",
        ".article_viewer",
        ".ArticleContentBox",
        "[class*='se-main-container']",
        "[class*='content']",
    ]

    for selector in paragraph_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                text = elem.text.strip()
                if text and len(text) >= 2:
                    paragraph_texts.append(text)
            if paragraph_texts:
                break
        except Exception:
            pass

    paragraph_texts = list(dict.fromkeys(paragraph_texts))
    content = "\n".join(paragraph_texts).strip()

    print("본문 길이:", len(content))
    if content:
        print("본문 미리보기:", content[:100])

    return content


def click_comment_more_buttons(driver, max_click=10):
    for _ in range(max_click):
        clicked = False
        try:
            buttons = driver.find_elements(By.TAG_NAME, "a") + driver.find_elements(By.TAG_NAME, "button")
            for btn in buttons:
                text = btn.text.strip()
                if any(key in text for key in ["댓글 더보기", "더보기", "답글 더보기"]):
                    try:
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.7)
                        clicked = True
                        break
                    except Exception:
                        pass
        except Exception:
            pass

        if not clicked:
            break


def get_article_comments(driver):
    comments = []

    # 댓글 영역까지 스크롤해서 댓글 HTML 로딩
    try:
        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 900);")
            time.sleep(0.6)
    except Exception:
        pass

    # 중요: '댓글' 버튼/링크 클릭 금지
    # 잘못 클릭하면 댓글 영역이 아니라 '댓글단 글' 같은 다른 페이지로 이동할 수 있음
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text("\n", strip=True)

    print("페이지에 '등록순' 포함 여부:", "등록순" in page_text)
    print("페이지에 'BEST 댓글' 포함 여부:", "BEST 댓글" in page_text)

    start_idx = -1
    for key in ["등록순", "최신순", "BEST 댓글"]:
        idx = page_text.find(key)
        if idx != -1 and (start_idx == -1 or idx < start_idx):
            start_idx = idx

    if start_idx == -1:
        print("댓글 영역 시작점을 못 찾음")
        print(page_text[:1200])
        return comments

    comment_area = page_text[start_idx:]
    print("댓글 영역 미리보기:")
    print(comment_area[:1500])

    lines = [x.strip() for x in comment_area.split("\n") if x.strip()]

    date_pat = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2}")

    stop_words = {
        "BEST",
        "BEST 댓글",
        "등록순",
        "최신순",
        "댓글알림",
        "클린봇",
        "답글쓰기",
        "신고",
        "공유",
        "댓글",
        "...",
        "♡",
        "♥",
    }

    used = set()

    for i, line in enumerate(lines):
        m = date_pat.search(line)
        if not m:
            continue

        comment_date_text = m.group(0)
        comment_date = parse_comment_date(comment_date_text)

        # 날짜 위쪽에서 댓글 본문 후보 찾기
        prev_lines = []
        j = i - 1

        while j >= 0:
            pl = lines[j].strip()

            if date_pat.search(pl):
                break

            pl = re.sub(r"^BEST\s*", "", pl).strip()

            if not pl or pl in stop_words:
                j -= 1
                continue

            if re.fullmatch(r"\d+", pl):
                j -= 1
                continue

            if pl.startswith("좋아요"):
                j -= 1
                continue

            if "님의 게시글 더보기" in pl:
                j -= 1
                continue

            prev_lines.append(pl)
            j -= 1

            if len(prev_lines) >= 10:
                break

        prev_lines.reverse()

        if not prev_lines:
            continue

        # 보통 구조:
        # 작성자명 / 등급숫자 / BEST / 댓글본문 / 날짜
        # 그래서 날짜 바로 위쪽의 긴 텍스트를 댓글 본문으로 우선 선택
        text_candidates = []

        for pl in prev_lines:
            if len(pl) >= 2:
                text_candidates.append(pl)

        if not text_candidates:
            continue

        # 가장 마지막 후보가 보통 댓글 본문
        comment_text = text_candidates[-1]

        # 여러 줄 댓글인 경우: 작성자명으로 보이는 첫 줄은 버리고 나머지를 합침
        if len(text_candidates) >= 2:
            possible_body = text_candidates[1:]
            joined_body = " ".join(possible_body).strip()

            # 두 줄 이상이고 본문처럼 보이면 합친 버전 사용
            if len(joined_body) > len(comment_text):
                comment_text = joined_body

        comment_text = re.sub(r"^BEST\s*", "", comment_text).strip()
        comment_text = re.sub(r"\s+", " ", comment_text).strip()
        comment_text = clean_text(comment_text)

        if not comment_text or len(comment_text) < 2:
            continue

        # BEST 댓글은 위에 한 번 더 뜨므로 본문+시간 기준 중복 제거
        key = (comment_text, comment_date_text)
        if key in used:
            continue

        used.add(key)

        comments.append({
            "comment_text": comment_text,
            "comment_date_text": comment_date_text,
            "comment_date": comment_date,
        })

    # BEST 위치 무시하고 실제 작성 시간순 정렬
    comments = sorted(
        comments,
        key=lambda x: x["comment_date"] if x["comment_date"] is not None else datetime.max
    )

    print("수집된 댓글 수:", len(comments))

    if comments:
        print("댓글 샘플:")
        for c in comments[:5]:
            print("-", c["comment_date_text"], c["comment_text"][:120])

    return comments

def get_next_article_url(driver):
    try:
        # 현재 프레임 HTML 기준으로 먼저 탐색
        html = driver.page_source

        if "다음글" in html:
            idx = html.find("다음글")
            print("현재 프레임 HTML에서 다음글 발견:")
            print(html[max(0, idx - 300): idx + 300])
        else:
            print("현재 프레임 HTML에 다음글 문자열 없음")

        soup = BeautifulSoup(html, "html.parser")

        # 1. 텍스트에 다음글이 들어간 a 태그 찾기
        for a in soup.find_all("a", href=True):
            text = a.get_text(" ", strip=True)
            attrs = " ".join(str(v) for v in a.attrs.values())

            if "다음글" in text or "다음글" in attrs:
                href = a.get("href")
                print("BeautifulSoup 다음글 후보 text:", text)
                print("BeautifulSoup 다음글 후보 href:", href)

                if href:
                    if href.startswith("http"):
                        return href
                    if href.startswith("/"):
                        return "https://cafe.naver.com" + href

        # 2. 못 찾으면 default_content HTML에서도 탐색
        driver.switch_to.default_content()
        html = driver.page_source

        if "다음글" in html:
            idx = html.find("다음글")
            print("default HTML에서 다음글 발견:")
            print(html[max(0, idx - 300): idx + 300])
        else:
            print("default HTML에 다음글 문자열 없음")

        soup = BeautifulSoup(html, "html.parser")

        for a in soup.find_all("a", href=True):
            text = a.get_text(" ", strip=True)
            attrs = " ".join(str(v) for v in a.attrs.values())

            if "다음글" in text or "다음글" in attrs:
                href = a.get("href")
                print("default BeautifulSoup 다음글 후보 text:", text)
                print("default BeautifulSoup 다음글 후보 href:", href)

                if href:
                    if href.startswith("http"):
                        return href
                    if href.startswith("/"):
                        return "https://cafe.naver.com" + href

        # 3. 그래도 못 찾으면 iframe들 전부 순회
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        print("iframe 순회 개수:", len(iframes))

        for idx, iframe in enumerate(iframes):
            try:
                driver.switch_to.default_content()
                driver.switch_to.frame(iframe)
                html = driver.page_source

                if "다음글" not in html:
                    continue

                print(f"iframe {idx} HTML에서 다음글 발견")
                soup = BeautifulSoup(html, "html.parser")

                for a in soup.find_all("a", href=True):
                    text = a.get_text(" ", strip=True)
                    attrs = " ".join(str(v) for v in a.attrs.values())

                    if "다음글" in text or "다음글" in attrs:
                        href = a.get("href")
                        print("iframe BeautifulSoup 다음글 후보 text:", text)
                        print("iframe BeautifulSoup 다음글 후보 href:", href)

                        if href:
                            if href.startswith("http"):
                                return href
                            if href.startswith("/"):
                                return "https://cafe.naver.com" + href

            except Exception as e:
                print(f"iframe {idx} 탐색 실패:", e)

        print("다음글 링크를 못 찾음")
        return None

    except Exception as e:
        print("다음글 URL 가져오기 실패:", e)
        return None

def save_cafe_checkpoint(posts, article_url, checked_count, force_excel=False):
    df = pd.DataFrame(posts)
    if df.empty:
        return df

    df = df.reindex(columns=CAFE_COLUMNS)
    df = df[df["text_clean"].notna()]
    df = df[df["text_clean"].astype(str).str.len() >= 2]
    df = df.drop_duplicates(subset=["text_clean", "content_type", "url"]).reset_index(drop=True)

    df.to_csv(CAFE_CHECKPOINT_CSV, index=False, encoding="utf-8-sig")

    progress = {
        "article_url": article_url,
        "checked_count": checked_count,
        "saved_count": len(df),
        "updated_at": datetime.now().isoformat(timespec="seconds"),
    }
    CAFE_PROGRESS_JSON.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding="utf-8")

    if force_excel:
        df.to_excel(CAFE_FINAL_XLSX, index=False)

    return df


def load_cafe_resume():
    posts = []
    checked_count = 0
    article_url = START_ARTICLE_URL

    if CAFE_CHECKPOINT_CSV.exists():
        try:
            old_df = pd.read_csv(CAFE_CHECKPOINT_CSV, encoding="utf-8-sig")
            posts = old_df.to_dict("records")
            print("기존 카페 체크포인트 로드:", old_df.shape)
        except Exception as e:
            print("체크포인트 로드 실패:", e)

    if CAFE_PROGRESS_JSON.exists():
        try:
            progress = json.loads(CAFE_PROGRESS_JSON.read_text(encoding="utf-8"))
            checked_count = int(progress.get("checked_count", 0))
            article_url = progress.get("article_url") or START_ARTICLE_URL
            print("진행상황 로드:", progress)
        except Exception as e:
            print("진행상황 로드 실패:", e)

    return posts, checked_count, article_url


def crawl_cafe_posts_and_comments(resume=True):
    if resume:
        posts, checked_count, article_url = load_cafe_resume()
    else:
        posts, checked_count, article_url = [], 0, START_ARTICLE_URL

    driver = create_driver()
    consecutive_errors = 0

    try:
        driver.get("https://nid.naver.com/nidlogin.login")
        print("네이버 로그인 후 이 셀의 input에 Enter를 누르세요.")
        input("로그인 완료 후 Enter: ")

        for i in range(checked_count, MAX_CHECK_POSTS):
            print("\n" + "=" * 60)
            print(f"{i + 1}번째 글 수집 시작")
            print("현재 article_url:", article_url)

            try:
                driver.switch_to.default_content()
                driver.get(article_url)
                time.sleep(5)

                entered = enter_cafe_main(driver)
                if not entered:
                    driver.switch_to.default_content()
                    print("cafe_main iframe 없음/진입 실패 → 현재 페이지에서 바로 수집")

                if i > 0 and i % RESTART_EVERY == 0:
                    print(f"{RESTART_EVERY}개 확인 완료. 현재 글 다시 로딩.")
                    driver.switch_to.default_content()
                    driver.get(article_url)
                    time.sleep(5)
                    enter_cafe_main(driver)

                date_text, article_date = get_article_date(driver)
                print("날짜 원문:", date_text)
                print("파싱 날짜:", article_date)

                title = get_article_title(driver)
                print("제목:", title)

                if article_date is None:
                    print("날짜 확인 실패. 저장하지 않고 다음글로 이동합니다.")

                elif article_date < START_DATE:
                    print("수집 시작일보다 과거 글에 도달하여 종료합니다.")
                    break

                elif START_DATE <= article_date <= END_DATE:
                    content = get_article_content(driver)

                    if clean_text(content):
                        posts.append({
                            "source": "cafe",
                            "round": ROUND,
                            "title": title,
                            "post_date": article_date,
                            "comment_date": None,
                            "text_original": content,
                            "text_clean": clean_text(content),
                            "content_type": "cafe_post",
                            "period": get_period(article_date),
                            "url": article_url,
                        })
                        print("본문 저장됨")

                    try:
                        comments = get_article_comments(driver)
                        for comment in comments:
                            comment_text = comment["comment_text"]
                            comment_date = comment["comment_date"] or article_date

                            posts.append({
                                "source": "cafe",
                                "round": ROUND,
                                "title": title,
                                "post_date": article_date,
                                "comment_date": comment_date,
                                "text_original": comment_text,
                                "text_clean": clean_text(comment_text),
                                "content_type": "cafe_comment",
                                "period": get_period(article_date),
                                "url": article_url,
                            })

                        print("댓글 저장됨:", len(comments))

                    except Exception as e:
                        print("댓글 수집 실패:", e)

                else:
                    print("날짜 범위 밖. 저장하지 않고 다음글로 이동합니다.")

                checked_count = i + 1
                df_checkpoint = save_cafe_checkpoint(
                    posts,
                    article_url,
                    checked_count,
                    force_excel=(i % 20 == 0),
                )
                print("현재 카페 저장 개수:", len(df_checkpoint) if df_checkpoint is not None else 0)

                next_url = get_next_article_url(driver)
                print("가져온 next_url:", next_url)

                if next_url is None:
                    print("다음글 URL 없음. 종료합니다.")
                    break

                if next_url == article_url:
                    print("다음글 URL이 현재 URL과 같습니다. 종료합니다.")
                    break

                article_url = next_url
                consecutive_errors = 0
                time.sleep(1.0)

            except Exception as e:
                consecutive_errors += 1
                print("글 수집 중 오류:", e)
                traceback.print_exc()
                save_cafe_checkpoint(posts, article_url, checked_count, force_excel=True)
                if consecutive_errors >= 5:
                    print("연속 오류 5회로 안전 종료합니다.")
                    break
                time.sleep(5)

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    final_cafe_df = save_cafe_checkpoint(posts, article_url, checked_count, force_excel=True)
    if final_cafe_df is None or final_cafe_df.empty:
        raise ValueError("카페 데이터가 수집되지 않았습니다.")

    final_cafe_df.to_csv(CAFE_FINAL_CSV, index=False, encoding="utf-8-sig")
    final_cafe_df.to_excel(CAFE_FINAL_XLSX, index=False)

    print("카페 수집 완료:", final_cafe_df.shape)
    print(CAFE_FINAL_CSV)
    print(CAFE_FINAL_XLSX)
    return final_cafe_df


# 네이버 로그인 후 Enter를 누르면 됩니다.
cafe_raw_df = crawl_cafe_posts_and_comments(resume=False)
display(cafe_raw_df.head())



끊겨서 저장파일 불러오려고

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(r"C:\Users\soyou\OneDrive\바탕 화면\textdata\arsenal_pipeline_outputs")
CAFE_FINAL_XLSX = OUTPUT_DIR / "arsenal_cafe_posts_comments.xlsx"

cafe_raw_df = pd.read_excel(CAFE_FINAL_XLSX)

print("cafe_raw_df shape:", cafe_raw_df.shape)
print(cafe_raw_df["content_type"].value_counts(dropna=False))
display(cafe_raw_df.head())

In [ ]:
# =====================================================
# 9. 카페 공통 포맷 변환
# =====================================================

def make_cafe_final(cafe_raw_df):
    cafe_raw_df = cafe_raw_df.copy()

    cafe_final = pd.DataFrame(index=cafe_raw_df.index)
    cafe_final["source"] = "cafe"
    cafe_final["round"] = cafe_raw_df["round"] if "round" in cafe_raw_df.columns else ROUND
    cafe_final["club"] = CLUB
    cafe_final["title"] = cafe_raw_df["title"]
    cafe_final["video_id"] = None
    cafe_final["video_title"] = None
    cafe_final["video_published_at"] = None
    cafe_final["channel_title"] = None

    cafe_final["content_type"] = cafe_raw_df["content_type"].map({
        "post": "cafe_post",
        "comment": "cafe_comment",
        "cafe_post": "cafe_post",
        "cafe_comment": "cafe_comment",
    }).fillna(cafe_raw_df["content_type"])

    cafe_final["text_original"] = cafe_raw_df["text_original"]
    cafe_final["text_clean"] = cafe_raw_df["text_clean"].fillna(cafe_raw_df["text_original"].apply(clean_text))
    cafe_final["content_published_at"] = to_naive_datetime_series(cafe_raw_df["post_date"])
    cafe_final["comment_published_at"] = to_naive_datetime_series(cafe_raw_df["comment_date"])
    cafe_final["like_count"] = None
    cafe_final["url"] = cafe_raw_df["url"]
    cafe_final["period"] = cafe_raw_df["period"] if "period" in cafe_raw_df.columns else cafe_final["content_published_at"].apply(get_period)
    cafe_final["gpt_sentiment"] = None
    cafe_final["gpt_confidence"] = None
    cafe_final["gpt_target"] = None
    cafe_final["gpt_reason"] = None
    cafe_final["topic"] = None

    return cafe_final.reindex(columns=FINAL_COLUMNS)


cafe_final = make_cafe_final(cafe_raw_df)
print("cafe_final shape:", cafe_final.shape)
print(cafe_final["content_type"].value_counts(dropna=False))
display(cafe_final.head())



In [ ]:
# =====================================================
# 10. YouTube + 카페 병합
# =====================================================

total_df = pd.concat(
    [youtube_final, cafe_final],
    ignore_index=True,
)

total_df = total_df[
    total_df["text_clean"].notna() &
    (total_df["text_clean"].astype(str).str.len() >= 2)
].reset_index(drop=True)

total_df["gpt_sentiment"] = total_df["gpt_sentiment"].apply(normalize_sentiment)

save_dataframe_both(total_df, MERGED_BEFORE_GPT_CSV)

print("통합 데이터:", total_df.shape)
print("\nsource 분포:")
print(total_df["source"].value_counts(dropna=False))
print("\n감정 분포:")
print(total_df["gpt_sentiment"].value_counts(dropna=False))
display(total_df.head())



In [ ]:
import os

from getpass import getpass
from openai import OpenAI
import pandas as pd

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=OPENAI_API_KEY)

work_df = pd.read_excel(REANALYSIS_CHECKPOINT_XLSX)

err_mask = work_df["gpt_sentiment"].astype(str).str.lower().eq("error")
nan_mask = work_df["gpt_sentiment"].isna()

retry_mask = err_mask | nan_mask

print("재분석 다시 돌릴 대상:", retry_mask.sum())
print(work_df.loc[retry_mask, "source"].value_counts(dropna=False))

work_df.loc[retry_mask, "_needs_reanalysis"] = True
work_df.loc[retry_mask, "_context_pass_done"] = False
work_df.loc[retry_mask, "_neutral_refine_done"] = False

In [ ]:
for col in ["_needs_reanalysis", "_context_pass_done", "_neutral_refine_done"]:
    work_df[col] = work_df[col].fillna(False).astype(bool)

print(work_df[["_needs_reanalysis", "_context_pass_done", "_neutral_refine_done"]].dtypes)
print("재분석 대상:", work_df["_needs_reanalysis"].sum())

In [ ]:
work_df = pd.read_excel(OUTPUT_FINAL_XLSX)

err_mask = work_df["gpt_sentiment"].astype(str).str.strip().str.lower().eq("error")
nan_mask = work_df["gpt_sentiment"].isna()
retry_mask = err_mask | nan_mask

work_df["_needs_reanalysis"] = False
work_df["_context_pass_done"] = True
work_df["_neutral_refine_done"] = True

work_df.loc[retry_mask, "_needs_reanalysis"] = True
work_df.loc[retry_mask, "_context_pass_done"] = False
work_df.loc[retry_mask, "_neutral_refine_done"] = False

print("재분석 다시 돌릴 대상:", retry_mask.sum())
print(work_df.loc[retry_mask, "source"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 11. 재분석용 GPT 함수
#     - 유튜브 1차 neutral + 카페 전체에 사용
# =====================================================

client = OpenAI(api_key=OPENAI_API_KEY)

def analyze_context_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""

너는 한국어 축구 팬 텍스트를 분석하는 target-aware 감정분석 분류기다.
분석 대상은 EPL 아스날 관련 YouTube 댓글과 네이버 카페 게시글/댓글이다.

[아스날 선수단 정보]
감독: 미켈 아르테타

선수:
가브리엘 마갈량이스, 가브리엘 마르티넬리, 가브리엘 제주스,
노니 마두에케, 다비드 라야, 데클란 라이스, 레안드로 트로사르,
리카르도 칼라피오리, 마르틴 수비멘디, 마르틴 외데고르,
마일스 루이스-스켈리, 막스 다우만, 말리 샐먼, 미켈 메리노,
벤 화이트, 부카요 사카, 빅토르 요케레스, 안드레 해리먼-아누스,
에베레치 에제, 위리엔 팀버르, 윌리엄 살리바, 이선 은와네리,
카이 하베르츠, 케파 아리사발라가, 크리스티안 뇌가르,
크리스티안 모스케라, 토미 셋포드, 피에로 인카피에, 야쿠브 키비오르

[아스날 선수/감독 반영 규칙]
- 아스날 선수나 미켈 아르테타를 칭찬하면 positive로 분류한다.
- 아스날 선수나 미켈 아르테타를 비판, 조롱, 비꼼, 불만 표현으로 언급하면 negative로 분류한다.
- 아스날 선수 이름이 직접 나오면 target은 보통 arsenal_player로 둔다.
- 아스날 이외의 다른 대상만 언급하면 아스날 감정으로 보지 않고 neutral로 분류한다.

[데이터 맥락]
{match_context}
위 데이터 맥락에는 아스날 선수단, 감독, 경기 결과, 라운드 정보가 포함되어 있다.
선수 이름, 감독 이름, 경기 결과, 라운드 맥락이 등장하면 반드시 반영한다.

[데이터 출처별 해석 규칙]

1. source가 youtube인 경우
- YouTube 댓글은 특정 영상 또는 경기 장면에 대한 즉각적 반응일 수 있다.
- 영상 제목, 라운드, 경기 결과, 스코어 맥락을 함께 반영한다.
- 댓글의 대상이 생략되어도 영상/경기 맥락상 아스날에 대한 반응으로 자연스럽게 해석되면 positive 또는 negative로 판단한다.
- 방송 출연진, 편집, 해설, 유튜버 자체에 대한 반응이면 아스날 감정이 아니므로 neutral 또는 broadcast로 둔다.
- 아스날이 승리한 경기에서는 "골 미쳤다", "마무리 좋다", "슈팅 지렸다", "수비 좋다"처럼 대상이 불명확한 경기력 칭찬은 특별히 상대팀 언급이 없으면 아스날 positive로 우선 판단한다. 아스날이 패배한 경기에서는 아쉬움, 비판, 조롱, 실망 표현이 아스날을 향하면 negative로 판단한다.
- 단, 패배 경기라도 아스날 선수 개인의 좋은 플레이를 칭찬하면 positive로 판단한다


2. source가 cafe인 경우
- 카페 게시글/댓글은 특정 경기 장면 반응으로 단정하지 않는다.
- 게시글 제목, 작성 시점, 38R 전후 맥락, 본문/댓글 전체 의미를 우선한다.
- 시즌 총평, 선수 평가, 감독/전술 평가, 이적시장 의견, 구단 운영 평가, 팬덤 여론이 중심일 수 있다.
- 단순 정보 공유, 질문, 링크 공유, 일정 확인은 neutral로 둔다.
- 아스날 팀/선수/감독/구단 운영/전술/시즌 결과에 대한 기대, 칭찬, 만족, 안도는 positive로 본다.
- 아스날 팀/선수/감독/구단 운영/전술/시즌 결과에 대한 조롱, 비판, 불만, 불안, 실망은 negative로 본다.

[분류 기준]
- positive: {CLUB_KO} 팀/선수/감독/전술/경기 결과/구단 운영에 대한 칭찬, 만족, 기대, 축하, 안도
- negative: {CLUB_KO} 팀/선수/감독/전술/경기 결과/구단 운영에 대한 비판, 조롱, 불만, 불안, 분노
- neutral: {CLUB_KO}와 직접 관련 없거나 단순 질문/정보/링크/일정 공유/감정 판단 불가

[중요 규칙]
- 대상이 {CLUB_KO}가 아니거나 불명확하면 neutral.
- 상대팀/상대선수 칭찬은 {CLUB_KO}와 무관하면 neutral.
- 상대팀 칭찬이 {CLUB_KO}의 약점/패배/부진을 강조하는 문맥이면 negative.
- YouTube 댓글은 영상 제목과 경기 맥락을 반영해 해석한다.
- 카페 글/댓글은 게시글 제목, 작성 시점, 본문 전체 의미를 우선한다.
- neutral은 감정 방향이 정말 없을 때만 사용한다.

[target 기준]
- arsenal_team
- arsenal_player
- arsenal_manager
- opponent
- referee_or_league
- broadcast
- unclear

[출력 규칙]
반드시 JSON 배열만 출력한다.
설명 문장, 코드블록, 마크다운은 출력하지 않는다.
각 결과에는 입력 번호 id를 포함한다.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": "arsenal_team 또는 arsenal_player 또는 arsenal_manager 또는 opponent 또는 referee_or_league 또는 broadcast 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[분석할 텍스트]
{comments_text}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 팬 댓글/게시글 감정분석 전문가다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "error",
                "confidence": 0.0,
                "target": "error",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]


def analyze_neutral_refine_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 아스날 관련 YouTube 댓글과 네이버 카페 게시글/댓글의 2차 감정분석 분류기다.
아래 텍스트들은 1차 분석에서 neutral로 분류되었거나, 아직 감정분석이 없는 카페 데이터다.
실제로 긍정/부정 뉘앙스가 숨어 있는지 재검토한다.

[데이터 맥락]
{match_context}

[가장 중요한 원칙]
- neutral은 감정 방향이 정말 없을 때만 유지한다.
- 하지만 모든 텍스트를 억지로 positive/negative로 바꾸지는 않는다.
- 아스날 팀/선수/감독/전술/구단 운영에 대한 칭찬, 기대, 만족, 안도, 축하가 조금이라도 있으면 positive.
- 아스날 팀/선수/감독/전술/구단 운영에 대한 비판, 조롱, 불만, 불안, 분노, 실망이 조금이라도 있으면 negative.
- 단순 정보/질문/링크/일정 공유/선수 이름만 언급한 글은 neutral.

[데이터 출처별 재검토 기준]

1. source가 youtube인 경우
- 유튜브 댓글은 특정 영상/경기 장면에 대한 짧은 반응일 수 있다.
- 댓글이 짧거나 대상이 생략되어도 영상/경기 맥락상 아스날에 대한 반응으로 자연스럽게 해석되면 positive 또는 negative로 분류한다.
- 방송 출연진, 편집, 해설, 유튜버 자체에 대한 반응이면 아스날 감정이 아니므로 neutral 또는 broadcast 대상으로 둔다.

2. source가 cafe인 경우
- 카페 게시글/댓글은 특정 경기 장면 반응으로 단정하지 않는다.
- 게시글 제목, 작성 시점, 38R 전후 맥락, 문장 전체 의미를 종합해 판단한다.
- 시즌 총평, 선수 평가, 감독/전술 평가, 이적시장 의견, 구단 운영 평가에 대한 감정이 있으면 positive 또는 negative로 분류한다.
- 단순 이적 루머 전달, 정보 공유, 질문, 링크 공유, 일정 확인, 선수 이름만 언급한 글은 neutral로 유지한다.

[neutral에서 바꿔야 하는 경우]
- "역시 아스날"처럼 칭찬 또는 비꼼/조롱으로 해석되면 문맥에 따라 positive 또는 negative.
- "이 정도면 괜찮다", "기대된다", "좋은 영입", "든든하다"처럼 긍정 평가가 있으면 positive.
- "답 없다", "불안하다", "또 망했다", "이러면 안 된다", "별로다"처럼 부정 평가가 있으면 negative.
- 상대팀 칭찬이라도 아스날의 패배, 약점, 부진, 비교 열세를 강조하면 negative.

[target 기준]
- arsenal_team
- arsenal_player
- arsenal_manager
- opponent
- referee_or_league
- broadcast
- unclear

[출력 규칙]
반드시 JSON 배열만 출력한다.
설명 문장, 코드블록, 마크다운은 출력하지 않는다.
각 결과에는 입력 번호 id를 포함한다.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": "arsenal_team 또는 arsenal_player 또는 arsenal_manager 또는 opponent 또는 referee_or_league 또는 broadcast 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[재검토할 텍스트]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 댓글 감정분석 전문가다. neutral 판정은 매우 보수적으로 사용한다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "neutral",
                "confidence": 0.0,
                "target": "unclear",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]


끊긴거 불러오기

In [ ]:
def add_temp_context(df):
    temp = df.copy()

    temp["_match_context"] = temp.apply(make_match_context, axis=1)

    temp["_context_key"] = (
        temp["source"].astype(str) + " | " +
        temp["round"].astype(str) + " | " +
        temp["period"].astype(str) + " | " +
        temp["title"].astype(str)
    )

    return temp

In [ ]:
# =====================================================
# 12. 유튜브 neutral/error + 카페 error 2차 감정분석
#     - 체크포인트 재시작 가능
#     - 체크포인트 저장은 10배치마다
# =====================================================

CHECKPOINT_EVERY_BATCHES = 10


def add_temp_context(df):
    temp = df.copy()

    temp["_match_context"] = temp.apply(make_match_context, axis=1)

    temp["_context_key"] = (
        temp["source"].astype(str) + " | " +
        temp["round"].astype(str) + " | " +
        temp["period"].astype(str) + " | " +
        temp["title"].astype(str)
    )

    return temp


def save_reanalysis_checkpoint(df):
    df.to_excel(REANALYSIS_CHECKPOINT_XLSX, index=False)
    print("재분석 체크포인트 저장:", REANALYSIS_CHECKPOINT_XLSX)


def load_or_create_reanalysis_work_df(total_df):
    if REANALYSIS_CHECKPOINT_XLSX.exists():
        try:
            work_df = pd.read_excel(REANALYSIS_CHECKPOINT_XLSX)
            print("재분석 체크포인트 로드:", work_df.shape)
            return work_df
        except Exception as e:
            print("재분석 체크포인트 로드 실패. 새로 시작:", e)

    work_df = total_df.copy()
    work_df["_needs_reanalysis"] = (
        work_df["gpt_sentiment"].isna() |
        work_df["gpt_sentiment"].astype(str).str.lower().isin(["neutral", "error"])
    )
    work_df["_context_pass_done"] = False
    work_df["_neutral_refine_done"] = False
    return work_df


def apply_results_to_work_df(work_df, batch_df, batch_results, prefix=""):
    result_map = {}

    for r in batch_results:
        try:
            result_map[int(r.get("id"))] = r
        except Exception:
            pass

    for local_id, (_, row) in enumerate(batch_df.iterrows()):
        original_idx = row["index"]
        r = result_map.get(local_id, {})

        sentiment = normalize_sentiment(r.get("sentiment", "error"))

        if sentiment not in ["positive", "negative", "neutral", "error"]:
            sentiment = "error"

        work_df.at[original_idx, "gpt_sentiment"] = sentiment
        work_df.at[original_idx, "gpt_confidence"] = r.get("confidence", 0.0)
        work_df.at[original_idx, "gpt_target"] = r.get("target", "unclear")
        work_df.at[original_idx, "gpt_reason"] = prefix + str(r.get("reason", ""))


def run_context_reanalysis_first_pass(work_df, batch_size=20):
    work_df = add_temp_context(work_df)

    for col in ["_needs_reanalysis", "_context_pass_done", "_neutral_refine_done"]:
        if col not in work_df.columns:
            work_df[col] = False
        work_df[col] = work_df[col].fillna(False).astype(bool)

    pending_mask = (
        work_df["_needs_reanalysis"].astype(bool) &
        ~work_df["_context_pass_done"].astype(bool)
    )

    pending_df = work_df[pending_mask].copy()

    print("맥락 기반 1차 재분석 대상:", pending_df.shape)

    if not pending_df.empty:
        print(pending_df["source"].value_counts(dropna=False))

    global_batch_no = 0

    for context_key, group_df in tqdm(pending_df.groupby("_context_key"), desc="맥락 기반 1차 재분석"):
        match_context = group_df["_match_context"].iloc[0]
        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            global_batch_no += 1

            batch_df = group_df.iloc[start:start + batch_size]
            comments = batch_df["text_clean"].fillna("").astype(str).tolist()

            batch_results = analyze_context_batch(comments, match_context)

            apply_results_to_work_df(
                work_df,
                batch_df,
                batch_results,
                prefix="[맥락 재분석] "
            )

            for _, row in batch_df.iterrows():
                work_df.at[row["index"], "_context_pass_done"] = True

            if global_batch_no % CHECKPOINT_EVERY_BATCHES == 0:
                save_reanalysis_checkpoint(
                    work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")
                )

            time.sleep(0.3)

    save_reanalysis_checkpoint(
        work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")
    )

    return work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")


def run_neutral_refinement(work_df, batch_size=20):
    work_df = add_temp_context(work_df)

    for col in ["_needs_reanalysis", "_context_pass_done", "_neutral_refine_done"]:
        if col not in work_df.columns:
            work_df[col] = False
        work_df[col] = work_df[col].fillna(False).astype(bool)

    pending_mask = (
        work_df["_needs_reanalysis"].astype(bool) &
        work_df["gpt_sentiment"].astype(str).str.lower().eq("neutral") &
        ~work_df["_neutral_refine_done"].astype(bool)
    )

    pending_df = work_df[pending_mask].copy()

    print("neutral 재검토 대상:", pending_df.shape)

    if not pending_df.empty:
        print(pending_df["source"].value_counts(dropna=False))

    global_batch_no = 0

    for context_key, group_df in tqdm(pending_df.groupby("_context_key"), desc="neutral 재검토"):
        match_context = group_df["_match_context"].iloc[0]
        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            global_batch_no += 1

            batch_df = group_df.iloc[start:start + batch_size]
            comments = batch_df["text_clean"].fillna("").astype(str).tolist()

            batch_results = analyze_neutral_refine_batch(comments, match_context)

            apply_results_to_work_df(
                work_df,
                batch_df,
                batch_results,
                prefix="[neutral 재검토] "
            )

            for _, row in batch_df.iterrows():
                work_df.at[row["index"], "_neutral_refine_done"] = True

            if global_batch_no % CHECKPOINT_EVERY_BATCHES == 0:
                save_reanalysis_checkpoint(
                    work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")
                )

            time.sleep(0.3)

    save_reanalysis_checkpoint(
        work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")
    )

    return work_df.drop(columns=["_match_context", "_context_key"], errors="ignore")


# =====================================================
# 실행부
# 중요:
# - 이미 work_df를 체크포인트/최종파일에서 불러오고
# - error/NaN 행만 _needs_reanalysis=True로 열어둔 뒤 실행할 것
# =====================================================

# work_df = load_or_create_reanalysis_work_df(total_df)

work_df = run_context_reanalysis_first_pass(work_df, batch_size=BATCH_SIZE)
work_df = run_neutral_refinement(work_df, batch_size=BATCH_SIZE)

print("2차 감정분석 완료")
print(work_df["gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 12-1. 남은 error 후처리
#      - 전체의 약 0.9%라 neutral로 처리
# =====================================================

err_mask = work_df["gpt_sentiment"].astype(str).str.lower().eq("error")

print("neutral 처리할 error 개수:", err_mask.sum())

work_df.loc[err_mask, "gpt_sentiment"] = "neutral"
work_df.loc[err_mask, "gpt_confidence"] = 0.0
work_df.loc[err_mask, "gpt_target"] = "unclear"
work_df.loc[err_mask, "gpt_reason"] = "[error 후처리] 재분석 실패 또는 파싱 실패로 neutral 처리"

print("\n후처리 후 감정분포:")
print(work_df["gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 13. 최종 컬럼만 남기기 + 저장
# =====================================================

final_df = work_df.copy()

# 재분석용 임시 컬럼 제거
final_df = final_df.drop(
    columns=[
        "_needs_reanalysis",
        "_context_pass_done",
        "_neutral_refine_done",
        "_match_context",
        "_context_key",
    ],
    errors="ignore"
)

# 혹시 없는 컬럼은 None으로 채워서 FINAL_COLUMNS에 맞춤
for col in FINAL_COLUMNS:
    if col not in final_df.columns:
        final_df[col] = None

final_df = final_df.reindex(columns=FINAL_COLUMNS)

# 날짜 컬럼 정리
for col in ["content_published_at", "comment_published_at", "video_published_at"]:
    if col in final_df.columns:
        final_df[col] = pd.to_datetime(
            final_df[col],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

final_df.to_csv(
    OUTPUT_FINAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

final_df.to_excel(
    OUTPUT_FINAL_XLSX,
    index=False
)

print("\n✅ 최종 데이터 저장 완료")
print("CSV:", OUTPUT_FINAL_CSV)
print("Excel:", OUTPUT_FINAL_XLSX)

print("\n최종 데이터 shape:", final_df.shape)
print("\n최종 컬럼:")
print(final_df.columns.tolist())

print("\n최종 감정분포:")
print(final_df["gpt_sentiment"].value_counts(dropna=False))

display(final_df.head())

